## <center><b>Western University</b></center>
## <center><b>Faculty of Engineering</b></center>
## <center><b>Department of Electrical and Computer Engineering</b></center>

# <center><b>MSE 3302B FW25: Sensors and Actuators</b></center>

<center>January 7, 2026</center>

Instructor: <a href="mailto:echen29@uwo.ca?Subject=MSE3302B%20Project">Prof. Elvis Chen, Ph.D., LEL</a>
- Department of Electrical and Computer Engineering
- Robarts Research Institute
- School of Biomedical Engineering
- Department of Medical Biophysics
- Department of Computer Science

## Vision-based Mechatronics for Playing Rock-Paper-Scissors-Minus-One game

***

## Table of Contents
- [Due date](#due-date)
- [Project Overview](#project-overview)
- [Game Play](#game-play)
- [Optimal Strategy](#optimal-strategy)
- [Game Theory](#game-theory)
- [Solving the General Game](#solving-the-general-game)
- [Learning Objectives](#learning-objectives)
- [Evaluation Criteria](#evaluation-criteria)
- [What to Submit](#what-to-submit)
- [Implementation Strategy](#implementation-strategies)
- [Grading](#grading)

***

## Due Date

11:59pm, March 27, 2026

- The OWL submission site will remain open until 11:59pm, March 30, 2026 for accommodations, without penalty.
- Submissions received after March 30, 2026 will incur a penalty of 10% per day

This project report is to be submitted as a [Jupyter Notebook Document](https://jupyter.org/) or as a [Jupyter Book 2](https://jupyterbook.org/stable/), with executable Python code, on OWL.

## Project Overview

An essential component of any mechatronic system is the ability to adapt to its environment. In this regard, real-time analysis and optimal strategy selection are essential for many real-world applications. This project explores the intersection of computer vision (CV), machine learning, and game theory, realized through a physical mechatronic system to play the game of Rock-Paper-Scissors-Minus-One (RPS-1).

RPS-1 is an extension of the [intransitive](https://en.wikipedia.org/wiki/Intransitive_game) [hand game](https://en.wikipedia.org/wiki/Hand_game) of [Rock Paper Scissors](https://en.wikipedia.org/wiki/Rock_paper_scissors) (RPS). In the standard RPS game, usually played between two players, each player simultaneously forms one of three shapes with an outstretched hand in the shape of **rock** (a closed fist), **paper** (a flat hand), and **scissors** (a fist with the index finger and middle finger extended, forming a *V*). The term *intransitive* means that the pairwise competitions between the strategies contain a circle.

![Intransitive Hand Game](https://upload.wikimedia.org/wikipedia/commons/6/67/Rock-paper-scissors.svg)
(Image courtesy of [Wikipedia](https://en.wikipedia.org/wiki/File:Rock-paper-scissors.svg))

RPS-1 is a strategic variant where two players simultaneously display two hands each (four hands total), then remove one hand each, with the winner determined by standard rock-paper-scissors rules applied to the remaining hands. This creates a complex decision space where optimal strategy depends on reading opponent intentions and maximizing win probability.

In the context of game theory, RPS and RPS-1 are [simultaneous](https://en.wikipedia.org/wiki/Simultaneous_game), [zero-sum](https://en.wikipedia.org/wiki/Zero-sum_game) games:
- A [**Simultaneous**](https://en.wikipedia.org/wiki/Simultaneous_game) or static game is a game where each player chooses their action without knowledge of the action chosen by other players,
- A [**Zero-Sum**](https://en.wikipedia.org/wiki/Zero-sum_game) game is a game where the result is an advantage for one side and an *equivalent loss* for the other.

## Game Play

The RPS-1 is a 2 player-game played in 2 stages:
1. Both players simultaneously show R, P, or S on each hand, then they say "minus one" and
1. Simultaneously pick one of their hands by withdrawing the other,

The outcome of the RPS-1 game is decided by the remaining hands from each player like the regular RPS game. In the event of a tie, they play again until someone wins.

The question we want to answer is, is there an **optimal** strategy for winning the RPS-1 game?

## Optimal Strategy

Is there a strategy for winning the RPS-1 game?

Let **R** denote a player playing *Rock*, **P** the *Paper*, and **S** the *Scissors*. At Stage-1 where each player plays with 2 hands, there are **9** possible combinations each player can play:
1. RR, Rock + Rock,
1. RP, Rock + Paper,
1. RS, Rock + Scissors,
1. PR, Paper + Rock,
1. PP, Paper + Paper,
1. PS, Paper + Scissors,
1. SR, Scissors + Rock,
1. SP, Scissors + Paper, and
1. SS, Scissors + Scissors

or we can reformat these combination of hand shapes into a matrix:

|  | Rock | Paper | Scissors |
| ----- | ----- | ----- | ----- |
| Rock | RR | RP | RS |
| Paper | PR | PP | PS |
| Scissors | SR | SP | SS |

But soon you will realize that you should **never** play both hands with the *same* shape (i.e. RR, PP, or SS), and the matrix is symmetrical (i.e. RP is the same play as PR).

Thus, out of the 9 possible combination one can play with 2 hands RPS, there are only *three* viable plays: RP, RS, and PS.

### Game Theory

To analyze the optimal strategies, consider
1. 1 point for a win,
1. 0 point for a tie, and
1. -1 point for a loss.

Assuming Player-1 plays **PS** (paper + Scissors) and Player-2 plays **PR** (Paper + Rock), then the **payoff** matrix for Player-1 for the matchup is:

|  | Paper | Rock  |
| ----- | ----- | ----- |
| Paper | 0 | 1  |
| Scissors | 1 | -1 |

You may infer, *if the objective of the game is **not to lose**, then Player-1 should always play paper*, as it ties against paper and wins against rock, so it never loses. However, **is this the best payoff**?

#### Payoff via equilibrium

The reason that Player-1 should not *always* play Paper in this scenario is that it does not maximize the payoff. Think about it: if Player-2 **expect** Player-1 to play Paper **always**, than Player-2 can simply play Paper as well to force a tie. Thus, while Player-1 will never lose, Player-1 will never win either!

**However**, if Player-1 **knows** that Player-2 is expecting Player-1 to play Paper, and thus Player-2 will play Paper accordingly, then Player-1 may want to play Scissors **expecting** Player-2 to play Paper, in order to score a win.

**But**... The same argument can be made for Player-2, who is expecting Player-1 to expect Player-2 to expect Player-1 to play Paper, to play Rock in anticapting Player-1 to play Scissors.

Thus, Player-1 should not play Paper **always**, and Player-2 should not expect Player-1 to do so either.

Suppose Player-1, instead of playing Paper **always**, picks Paper with a probability of $p$ and Scissors with a probability $1-p$. If Player-2 plays Paper, then the expected payoff is

$$
p \cdot 0 + (1-p) \cdot 1 \\
1 - p
$$

If Player-2 plays Rock, then the expected payoff for Player-1 is:

$$
p \cdot 1 + (1-p) \cdot -1 =\\
2 p -1
$$

Since Player-2 can only play either paper or rock, [Nash Equilibrium](https://en.wikipedia.org/wiki/Nash_equilibrium) dictates the total expected payoffs to Player-2 strategies must be equal:

$$
p \cdot 0 + (1-p) \cdot 1 = p \cdot 1 + (1-p) \cdot -1 \\
1-p = 2p-1 \\
p = 2/3
$$

Thus, **Player-1 should pick Paper with a $2/3$ probability and Scissors with a $1/3$ probability**! 

#### What about Player-2?

This is left as an exercise: if Player-2 plays paper with probability $q$ and rock with probability $(1-q)$, you can show that Player-2 should play Paper with a probability of $2/3$ and rock with a probability of $1/3$.

***

## Solving the general game

Refer to the lecture material presented on January 9, 2026 (Week 1, Lecture 2) for background materials. While you should complete the derivation of an optimal strategy, if you follow the previous derivation, you can conclude an optimal strategy as:

1. Play PR, SR, SP with equal probability
1. If your opponent foolishly plays a weakly dominated strategy, i.e. two identical hands such as RR, PP, and SS,
    - If one of your hands match, you are Guarantee a tie,
    - If both of your hands do not match, you are guarantee a win,
1. If your opponent plays the same strategy as you do, pick the stronger hand out of the two options,
    - e.g. If both of you play PR, play P,
1. If your opponent plays a different strategy as you do, one of your hand will match theirs:
    - Play that option with a $\frac{2}{3}=66\%$ probability and
    - Play the other option with a $\frac{1}{3}=33\%$ probability

***

## Learning Objectives

Students will:
- Design and implement a multi-degree-of-freedom articulated hand system,
- Implement a complete mechatronic system that integrates computer vision, decision-making logic, and electromechanical control,  
- Characterize and optimize actuator performance.
- Apply systems engineering principles to specify, design, and validate a mechatronic system that meets functional requirement, operational constraints, and performance metrics.

***

## Evaluation Criteria
This project uses Graduate Attributes (GAs) for assessment. Please note:

- This project is worth 30% of your final mark
- Total marks: 20 points
- Each Graduate Attribute (GA) is evaluated on a scale of 1 to 4
- Evaluation focuses on your **understanding** and **effort** in problem-solving
- The performance of the final mechatronic system is not the primary evaluation criterion, though consistent good performance may earn bonus marks
- Pay attention to the following rubric and specification of the project report in order to understand what is asked of you.


| Graduate Attribute | Unacceptable (1) | Below Expectations (2) | Meets Expectations (3) | Exceeds Expectations (4) |
|-------------------|-------------------|----------------------|----------------------|-------------------------|
| **KB4 - Specialized Engineering Knowledge** | Unable to articulate specialized discipline-specific engineering knowledge | Recollection of specialized discipline-specific engineering knowledge is inaccurate – fundamentals clearly wrong | Competent in the essential aspects of discipline-specific engineering knowledge | Meets expectations plus: Recognizes nuances of discipline-specific engineering knowledge |
| **ET1 - Engineering Tools** | Applicability of an ET to a problem not recognized: ET not used! | Improper ET selected because underlying mathematical AND/OR engineering fundamentals not recognized | ET selection justified by comprehension of the underlying mathematical AND/OR engineering fundamentals | Meets expectation, plus: Limitations of the selected tool are identified and assessed |
| **CS3 - Written Communication** | Technical language is unclear, with much jargon, presentation is incoherent, spelling grammar and syntax mostly incorrect, graphical tools not used where appropriate | Technical language is sometimes unclear or inappropriate to the audience, ideas and arguments are unpersuasive; spelling grammar and syntax interfere with understanding; graphical tools used but only partly effective | Technical language is clear, and appropriate to the audience, ideas are persuasive, spelling, grammar and syntax are mostly correct, graphical tools are used effectively | Meets expectations plus: Written document is sufficiently engaging to be understood and interesting to readers from other disciplines |
| **I3 - Data Analysis** | No meaningful analysis of data, leading to invalid conclusions | Conclusions are not justified by the data analysis presented OR experimental error and its impact not quantified | Conclusions justified by appropriate data analysis AND quantification of experimental error and its impact | Meets expectations, plus: Conclusions are framed in the context of prior knowledge and recommendations for future work are provided |
| **IESE1 - Societal Impact** | No awareness of impact of engineering on economic aspects of society | Unable to analyse effectively the impact of engineering on the society and environment | Able to quantify accurately the impact of engineering on the society and environment | Meets expectations plus: Analysis takes into account the uncertainty of prediction of interaction with society and environment |

***

## What to Submit

Your project will be evaluated through a Project Report submitted as a [Jupyter Notebook](https://jupyter.org/) or [Jupyterbook V2](https://jupyterbook.org/stable/) document. The Project Report, in addition to a detailed documentation, must be accompanied with a **complete system design materials (CAD drawings, source codes, etc.,)** for the instructor to verify your implementation.

### Project Submission Requirements

The Project Submission must include:

1. A comprehensive written report documenting your implementation
2. Complete and all materials related to the design of the mechatronic systems including, but not limited to:
   - CAD drawing
   - Source codes
   - Simulation (if any) codes and visualization
   - Include setup/installation instructions 
3. A *user manual* with instruction on how to use/deploy the mechatronic system (that is different than the setup/installation instructions)

The idea is that, if there is a group of students who wants to extend your implementation, they will have sufficient background materials to work with without duplication.

Your notebook should enable the instructor to:
* Understand your technical approach
* Run your implementation
* Verify your results
* Assess your learning outcomes


### Written Report Format

Your project report should contain these sections:

- **Introduction**: Detail the motivation and background context explaining why the game of RPS-1 was chosen
- **Methods**: Explain how *a* strategy for playing RPS-1 was developed. If a machine-learning (ML) model was used for image processing (e.g. recognizing hand shapes), explain why a particular model was chosen, what was needed/modified to implement for your application, and other technical details that may be needed for others to **reproduce** your work.
- **Results**: Write as a user manual explaining how to use your code. Include:
  * Sample test images
  * A set of documented tests
  * Results in tables and graphs
    * Consider using [numpy](https://numpy.org/) for any mathematical calculation including statistics
    * Consider using [matplotlib](https://matplotlib.org/) for graphs and plots.
    * The combination of [numpy](https://numpy.org/) and [matplotlib](https://matplotlib.org/) simulates the Matlab environment and the syntax are almost identical!
- **Discussion**: Explain:
  * Why you chose your approach
  * What was needed to make it work and why
  * If issues arose, document the reasons (**It is acceptable if your program doesn't work perfectly** as long as you investigate why)
  * Potential societal/economic implications and impact
- **Conclusion**: Detail what you learned
- **Bibliography**: Provide a list of references relevant to your report. For example, if you use a publicly available ML model, provide a citation/reference to the original paper and the URL where you downloaded the model.

### What to Upload to OWL

Each group should upload to OWL a single compressed (zip) file containing all the requested materials as outlined above.

## Implementation Strategies

Your objective is to design, integrate, and manufacture **two** hands with articulated fingers so that each articulated hands can change shapes between Rock, Paper, and Scissors. The UWO MSE lab has some manufacturing capability including 3D printing and laser cuttings that are available to you, and MSE students have acquired hardware kits for engineering courses that can be repurposed for this project.

- To design hands with articulated fingers, you may start with websites where 3D models are freely available, here is [one example](https://makerworld.com/en/models/1372777-articulated-hand#profileId-1419528),
- You are free to decide on the mechanism by which the hand is articulated. They may include, but not limited to
    - String pulley
    - Direct drive using motor
    - Geared system
- Sensing is achieved using computer vision (CV), i.e. a web-cam
    - You can assume that a USB camera will be made available to you during the demostration, or
    - You are free to integrate any camera system of your choosing (but at your own cost)
    - A table (i.e. flat surface) will be set up, with masking tapes defining 2 rectangular regions of the table for which each player must place their two-hands onto.
    - Two web-cams will be placed on top of the table, taking (static) pictures of the marked table.
    - You can assume that the two hands that are "up-side up" are your hands, and the other 2 hands are of that of an opponent.
- Between the stage-1 (4 hands) and the stage-2 (withdrwing one hand) of the game, your mechatronic system must change the hand shape in a pre-defined time limit.
- You can assume a computer-vision based software that recognizes the hand shapes to be available to you,
    - You are free to tweak it
    - You are free to implement your own approach
- You are free to come up with any gaming strategy, i.e. you do not have to use what was shown in the class/documents above, as long as you justify your gaming strategy in a logical manner.

***

## Grading

The grading scheme is based on the Rubrics shown above. Thus, it is important for you to understand what the Rubrics are asking for. Keep in mind that I am looking for the following
- Demonstration of understanding
  - For example, the detailed explanation of the optimal strategy beyond what was covered in the lecture,
- Demonstration of technical proficiency
  - Writing quality
  - Usage of Jupyter Notebook/Jupyter Book V2
- Demonstration of technical accuracy
  - Accuracy of object recognition (hands) and classification (hand shapes)
- Demonstration of self-reflection
  - Tell me about your journey of completing this project.
  - What did you learn, both on the technical and non-technical aspects?
  - What was the difficult part(s) and how did you overcome it?
- Social implication
  - How do you apply what you have learned throughout this project to a wider perspective?